# Constructing an LBO Model in Python

## Learning objectives

By the end of this notebook, you should be able to:

- Structure an LBO model with explicit assumptions.
- Build a transparent sequence from operating forecasts to equity outcomes.
- Compute MOIC and IRR in Python.
- Use sensitivity analysis to identify key return drivers.

## Where this notebook fits in the course

Notebook 1 focused on market-data workflow discipline. Here we apply the same mindset to corporate finance modeling. Notebook 3 will then move to market-level strategy backtesting.

## Notebook narrative

We proceed in the same logic used by investment teams:

1. Define assumptions and model architecture.
2. Project operations and cash generation.
3. Model debt evolution under cash flow constraints.
4. Calculate exit equity value and investor returns.
5. Stress-test results with sensitivity analysis.


## Why Python is useful for financial modeling

Python complements spreadsheet work by making model logic explicit, reusable, and scenario-friendly. This is especially useful for teaching because each modeling block can be tested and discussed independently.


## Libraries used and why

- `pandas`: projection tables and scenario outputs.
- `numpy`: numerical calculations and array operations.
- `numpy_financial`: IRR calculation.
- `matplotlib` and `seaborn`: sensitivity visualization.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import numpy_financial as npf
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titleweight"] = "bold"
pd.set_option("display.float_format", "{:.4f}".format)


## 1. Model architecture overview

**Flow:** inputs -> operating model -> debt schedule -> exit value -> equity returns

This modular structure improves readability and makes sensitivity analysis straightforward.


## 2. Input assumptions

The model is simplified for teaching, but assumptions are chosen to remain economically plausible.


In [ ]:
assumptions = {
    "entry_ebitda": 250.0,          # USDm
    "entry_multiple": 10.0,         # EV / EBITDA
    "revenue_growth": 0.08,         # annual growth
    "ebitda_margin": 0.25,          # EBITDA / Revenue
    "tax_rate": 0.25,               # effective tax rate
    "capex_pct_revenue": 0.04,      # CAPEX / Revenue
    "debt_ratio": 0.60,             # Debt / EV at entry
    "interest_rate": 0.07,          # annual cash interest
    "holding_period": 5,            # years
    "exit_multiple": 11.0,          # EV / EBITDA at exit
}

assumptions_df = pd.DataFrame.from_dict(assumptions, orient="index", columns=["Value"])
assumptions_df


### Discussion

Keeping assumptions centralized is good practice. It makes model governance easier and allows fast, transparent scenario testing.

### Section transition

With assumptions fixed, we build the operating layer first and keep financing logic separate.


## 3. Build the operating model

We project revenue, EBITDA, taxes, CAPEX, and pre-debt free cash flow over the holding period.


In [ ]:
years = np.arange(1, assumptions["holding_period"] + 1)

entry_revenue = assumptions["entry_ebitda"] / assumptions["ebitda_margin"]
revenue = [entry_revenue * (1 + assumptions["revenue_growth"]) ** y for y in years]
ebitda = [r * assumptions["ebitda_margin"] for r in revenue]
taxes = [e * assumptions["tax_rate"] for e in ebitda]
capex = [r * assumptions["capex_pct_revenue"] for r in revenue]
pre_debt_fcf = [e - t - c for e, t, c in zip(ebitda, taxes, capex)]

operating = pd.DataFrame({
    "Year": years,
    "Revenue": revenue,
    "EBITDA": ebitda,
    "Taxes": taxes,
    "CAPEX": capex,
    "PreDebtFCF": pre_debt_fcf,
})
operating


### Discussion

Modeling operations first clarifies whether value creation comes from business performance or financial structuring.

### Section transition

Next, we connect operating cash generation to debt amortization.


## 4. Build the debt schedule

Debt dynamics are driven by:

- opening debt,
- annual interest expense,
- available cash for principal repayment,
- closing debt.


In [ ]:
entry_ev = assumptions["entry_ebitda"] * assumptions["entry_multiple"]
initial_debt = entry_ev * assumptions["debt_ratio"]
entry_equity = entry_ev - initial_debt

debt_rows = []
opening_debt = initial_debt

for _, row in operating.iterrows():
    interest = opening_debt * assumptions["interest_rate"]
    cash_after_interest = max(row["PreDebtFCF"] - interest, 0)
    principal_repayment = min(cash_after_interest, opening_debt)
    closing_debt = opening_debt - principal_repayment

    debt_rows.append({
        "Year": int(row["Year"]),
        "OpeningDebt": opening_debt,
        "InterestExpense": interest,
        "PrincipalRepayment": principal_repayment,
        "ClosingDebt": closing_debt,
    })
    opening_debt = closing_debt

debt_schedule = pd.DataFrame(debt_rows)
debt_schedule


### Discussion

Faster debt reduction usually increases equity value at exit. This is why cash conversion is central in LBO underwriting.

### Section transition

Once debt evolution is known, we can estimate exit value and investor returns.


## 5. Exit valuation and equity returns


In [ ]:
exit_ebitda = operating.loc[operating["Year"] == assumptions["holding_period"], "EBITDA"].iloc[0]
exit_ev = exit_ebitda * assumptions["exit_multiple"]
exit_debt = debt_schedule.loc[debt_schedule["Year"] == assumptions["holding_period"], "ClosingDebt"].iloc[0]
exit_equity = exit_ev - exit_debt

cash_flows_equity = [-entry_equity] + [0] * (assumptions["holding_period"] - 1) + [exit_equity]
moic = exit_equity / entry_equity
irr = npf.irr(cash_flows_equity)

summary = pd.DataFrame({
    "Metric": ["Entry EV", "Entry Debt", "Entry Equity", "Exit EV", "Exit Debt", "Exit Equity", "MOIC", "IRR"],
    "Value": [entry_ev, initial_debt, entry_equity, exit_ev, exit_debt, exit_equity, moic, irr],
})
summary


### Discussion

This decomposition separates return sources: operating growth, deleveraging, and exit multiple effects.

### Section transition

To understand robustness, we now stress-test two major assumptions simultaneously.


## 6. Sensitivity analysis

We evaluate IRR sensitivity to:

- Exit multiple,
- EBITDA growth (modeled through revenue growth).


In [ ]:
def run_lbo_case(growth_rate: float, exit_multiple: float) -> float:
    years_local = np.arange(1, assumptions["holding_period"] + 1)
    revenue_local = [entry_revenue * (1 + growth_rate) ** y for y in years_local]
    ebitda_local = [r * assumptions["ebitda_margin"] for r in revenue_local]
    taxes_local = [e * assumptions["tax_rate"] for e in ebitda_local]
    capex_local = [r * assumptions["capex_pct_revenue"] for r in revenue_local]
    pre_debt_fcf_local = [e - t - c for e, t, c in zip(ebitda_local, taxes_local, capex_local)]

    debt_local = initial_debt
    for fcf in pre_debt_fcf_local:
        interest_local = debt_local * assumptions["interest_rate"]
        repay_local = min(max(fcf - interest_local, 0), debt_local)
        debt_local -= repay_local

    exit_equity_local = ebitda_local[-1] * exit_multiple - debt_local
    cash_flows_local = [-entry_equity] + [0] * (assumptions["holding_period"] - 1) + [exit_equity_local]
    return npf.irr(cash_flows_local)

exit_multiples = np.arange(8.0, 13.0, 1.0)
growth_rates = np.arange(0.04, 0.13, 0.02)

sensitivity = pd.DataFrame(
    index=[f"{g:.0%}" for g in growth_rates],
    columns=[f"{m:.1f}x" for m in exit_multiples],
    dtype=float,
)

for g in growth_rates:
    for m in exit_multiples:
        sensitivity.loc[f"{g:.0%}", f"{m:.1f}x"] = run_lbo_case(g, m)

sensitivity


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    sensitivity * 100,
    annot=True,
    fmt=".1f",
    cmap="YlGnBu",
    linewidths=0.3,
    cbar_kws={"label": "IRR (%)"},
    ax=ax,
)
ax.set_title("IRR Sensitivity: Exit Multiple vs EBITDA Growth")
ax.set_xlabel("Exit Multiple")
ax.set_ylabel("EBITDA Growth")
plt.tight_layout()
plt.show()


## 7. Interpretation

- Exit valuation assumptions can materially change equity outcomes.
- Growth and deleveraging reinforce each other in leveraged structures.
- Sensitivity tables are essential because single-point LBO outputs can be misleading.

### Discussion

The teaching objective is not perfect deal realism. It is disciplined model architecture that makes economic drivers explicit and debatable.


## Key takeaways

- A modular Python model makes LBO logic transparent and easier to audit.
- Return analysis is most useful when linked to clearly stated assumptions.
- Sensitivity analysis turns a static model into a decision-support tool.

## Bridge to Notebook 3

Notebook 2 modeled value creation at the company and capital-structure level. Notebook 3 shifts to market-level decision systems, where we map indicators to trading signals, backtest rules, and evaluate performance credibility.
